In [ ]:
#BLOCK-1
#---------------------------------------------------------------------------

from google.colab import drive
import os

drive.mount('/content/drive')

os.makedirs("/content/dataset", exist_ok=True)
print("Extracting datasets...")
!unzip -q "/content/drive/MyDrive/FinalDataset_PA.zip" -d "/content/dataset"
!unzip -q "/content/drive/MyDrive/Xray-temp-master.zip" -d "/content/"
print("Extraction completed.")

In [ ]:
#BLOCK-2
#---------------------------------------------------------------------------

import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split

csv_path = "/content/dataset/FinalDataset_PA/metadata_pa_only.csv"
df = pd.read_csv(csv_path)

# remove No Finding labels
clean_df = df[['Image Index', 'Finding Labels']].copy()
clean_df['Finding Labels'] = clean_df['Finding Labels'].apply(
    lambda x: [label for label in x.split('|') if label != "No Finding"] if isinstance(x, str) else []
)

# binarize labels
mlb = MultiLabelBinarizer()
binary_labels = mlb.fit_transform(clean_df['Finding Labels'])
labels_df = pd.DataFrame(binary_labels, columns=mlb.classes_)
final_df = pd.concat([clean_df['Image Index'], labels_df], axis=1)

# train/val/test split (80/10/10)
train_val, test_df = train_test_split(final_df, test_size=0.10, random_state=42)
train_df, val_df = train_test_split(train_val, test_size=0.1111, random_state=42)

train_df.to_csv("/content/train_split.csv", index=False)
val_df.to_csv("/content/val_split.csv", index=False)
test_df.to_csv("/content/test_split.csv", index=False)

print(f"Total classes: {len(mlb.classes_)}")
print(f"Splits - Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
#BLOCK-3
#---------------------------------------------------------------------------

import os

PROJECT_ROOT = "/content/Xray-temp-master/Xray-temp-master"
target_file = os.path.join(PROJECT_ROOT, "data/dataset.py")

dataset_code = """
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
import pandas as pd
import numpy as np

class LargeImageDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform
        self.img_names = df.iloc[:, 0].values
        self.labels = df.iloc[:, 1:].apply(pd.to_numeric, errors='coerce').fillna(0).values.astype(np.float32)

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_name = self.img_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        try:
            image = Image.open(img_path).convert("RGB")
        except:
            image = Image.new('RGB', (512, 512))
        
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(self.labels[idx])
"""

with open(target_file, "w") as f:
    f.write(dataset_code)
print("Dataset class updated.")

In [ ]:
#BLOCK-4
#---------------------------------------------------------------------------

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
import pandas as pd
import numpy as np
import os
import sys
from tqdm import tqdm
from sklearn.metrics import f1_score

PROJECT_ROOT = "/content/Xray-temp-master/Xray-temp-master"
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from models.full_model import DenseNetCBAM
from data.dataset import LargeImageDataset

# asymmetric loss definition
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=0, clip=0.05, eps=1e-8):
        super(AsymmetricLoss, self).__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, x, y):
        xs_pos = torch.sigmoid(x)
        xs_neg = 1 - xs_pos
        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1)
        loss_pos = y * torch.log(xs_pos.clamp(min=self.eps)) * (1 - xs_pos)**self.gamma_pos
        loss_neg = (1 - y) * torch.log(xs_neg.clamp(min=self.eps)) * (1 - xs_neg)**self.gamma_neg
        return -(loss_pos + loss_neg).mean()

# finding best threshold for each class
def get_optimal_thresholds(y_true, y_probs):
    num_classes = y_true.shape[1]
    best_thresholds = np.ones(num_classes) * 0.5
    for i in range(num_classes):
        best_f1_val = 0
        for thresh in np.arange(0.1, 0.85, 0.05):
            preds = (y_probs[:, i] > thresh).astype(int)
            f1 = f1_score(y_true[:, i], preds, zero_division=0)
            if f1 > best_f1_val:
                best_f1_val = f1
                best_thresholds[i] = thresh
    return best_thresholds

# config
EPOCHS = 30
BATCH_SIZE = 8
LR = 5e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TRAIN_CSV = "/content/train_split.csv"
VAL_CSV = "/content/val_split.csv"
IMG_DIR = "/content/dataset/FinalDataset_PA"

# transforms
train_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(7),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# loaders
train_ds = LargeImageDataset(TRAIN_CSV, IMG_DIR, train_transform)
val_ds = LargeImageDataset(VAL_CSV, IMG_DIR, val_transform)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# model setup
num_classes = len(pd.read_csv(TRAIN_CSV).columns) - 1
model = DenseNetCBAM()
if hasattr(model, 'classifier'):
    model.classifier = nn.Linear(model.classifier.in_features, num_classes)
elif hasattr(model, 'fc'):
    model.fc = nn.Linear(model.fc.in_features, num_classes)

model.to(DEVICE)
criterion = AsymmetricLoss(gamma_neg=4, gamma_pos=0, clip=0.05)
optimizer = optim.Adam(model.parameters(), lr=LR)

# train loop
os.makedirs("checkpoints", exist_ok=True)
print(f"Starting Training: ASL + Dynamic Thresholds on {DEVICE}")
best_macro_f1 = 0.0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{EPOCHS}]")

    for images, labels in loop:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    model.eval()
    val_loss = 0.0
    all_val_probs, all_val_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            v_loss = criterion(outputs, labels)
            val_loss += v_loss.item()
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_val_probs.append(probs)
            all_val_labels.append(labels.cpu().numpy())

    all_val_probs = np.concatenate(all_val_probs)
    all_val_labels = np.concatenate(all_val_labels)

    # find optimal thresholds per epoch
    current_thresholds = get_optimal_thresholds(all_val_labels, all_val_probs)
    val_preds = np.zeros_like(all_val_probs)
    for i in range(num_classes):
        val_preds[:, i] = (all_val_probs[:, i] > current_thresholds[i]).astype(int)

    epoch_f1 = f1_score(all_val_labels, val_preds, average='macro', zero_division=0)

    print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f} | Macro F1: {epoch_f1:.4f}")

    if epoch_f1 > best_macro_f1:
        best_macro_f1 = epoch_f1
        torch.save({
            'state_dict': model.state_dict(),
            'thresholds': current_thresholds,
            'best_f1': best_macro_f1,
            'epoch': epoch+1
        }, "checkpoints/absolute_best_model.pth")
        print(">>> Absolute best model updated.")

In [ ]:
#BLOCK-5
#---------------------------------------------------------------------------

import torch
import pandas as pd
import numpy as np

MODEL_PATH = "checkpoints/absolute_best_model.pth"

try:
    checkpoint = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
    best_thresholds = checkpoint['thresholds']
    best_f1 = checkpoint['best_f1']
    best_epoch = checkpoint.get('epoch', 'N/A')

    TRAIN_CSV = "/content/train_split.csv"
    class_names = pd.read_csv(TRAIN_CSV).columns[1:].tolist()

    print(f"MODEL ANALYSIS REPORT")
    print(f"Macro F1 Score : {best_f1:.4f}")
    print(f"Saved Epoch    : {best_epoch}")
    print("-" * 55)
    print(f"{'Pathology':<20} | {'Optimum Threshold':<25}")
    print("-" * 55)

    for name, thresh in zip(class_names, best_thresholds):
        print(f"{name:<20} | {float(thresh):.2f}")

    print("-" * 55)
except FileNotFoundError:
    print("Checkpoint file not found.")

In [ ]:
#BLOCK-6
#---------------------------------------------------------------------------

import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from torch.utils.data import DataLoader
from torchvision import transforms
import sys

MODEL_PATH = "checkpoints/absolute_best_model.pth"
TEST_CSV = "/content/test_split.csv"
IMG_DIR = "/content/dataset/FinalDataset_PA"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PROJECT_ROOT = "/content/Xray-temp-master/Xray-temp-master"
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from models.full_model import DenseNetCBAM
from data.dataset import LargeImageDataset

# load checkpoint and thresholds
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
best_thresholds = checkpoint['thresholds']

df_test = pd.read_csv(TEST_CSV)
CLASS_NAMES = df_test.columns[1:].tolist()
num_classes = len(CLASS_NAMES)

model = DenseNetCBAM()
if hasattr(model, 'classifier'):
    model.classifier = torch.nn.Linear(model.classifier.in_features, num_classes)
elif hasattr(model, 'fc'):
    model.fc = torch.nn.Linear(model.fc.in_features, num_classes)

model.load_state_dict(checkpoint['state_dict'])
model.to(DEVICE).eval()

test_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_ds = LargeImageDataset(TEST_CSV, IMG_DIR, test_transform)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)

all_probs, all_labels = [], []
print("Evaluating on test set with optimized thresholds...")
with torch.no_grad():
    for images, labels in tqdm(test_loader):
        images = images.to(DEVICE)
        outputs = model(images)
        probs = torch.sigmoid(outputs).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())

all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)

report_data = []
for i, name in enumerate(CLASS_NAMES):
    auc = roc_auc_score(all_labels[:, i], all_probs[:, i]) if len(np.unique(all_labels[:, i])) > 1 else 0.5
    current_thresh = best_thresholds[i]
    preds = (all_probs[:, i] > current_thresh).astype(int)
    f1 = f1_score(all_labels[:, i], preds, zero_division=0)
    acc = accuracy_score(all_labels[:, i], preds)

    report_data.append({
        "Pathology": name,
        "Threshold": current_thresh,
        "AUC": auc,
        "F1": f1,
        "Accuracy": acc
    })

df_report = pd.DataFrame(report_data)
mean_row = df_report.mean(numeric_only=True).to_dict()
mean_row["Pathology"] = "OVERALL (MACRO)"
mean_row["Threshold"] = np.nan
df_report = pd.concat([df_report, pd.DataFrame([mean_row])], ignore_index=True)

print("\nFINAL PERFORMANCE REPORT (OPTIMIZED THRESHOLDS)")
print("-" * 75)
print(df_report.to_string(index=False, float_format=lambda x: "{:.4f}".format(x)))
print("-" * 75)

df_report.to_csv("final_report_optimized_thresholds.csv", index=False)